# Pareto Multi-Mode MAPPO — Crop Disease UAV
Comparative baseline adapted from: *LLM-Driven Pareto-Optimal Multi-Mode RL*.

Three specialized PPO policies (Aggressive / Balanced / Cautious) with mode-specific
reward scaling. A hard-coded telemetry threshold selector (standing in for the paper's
LLM selector) chooses a mode at the start of each episode from disease pressure,
weather risk, and fleet battery context.
All policies share one centralized critic with per-UAV heads.
Environment: `uav_env_5` (identical disease dynamics to the main MAPPO baseline).

In [ ]:
import os, sys
ON_KAGGLE = os.path.exists('/kaggle/input')
if ON_KAGGLE:
    MODULE_DIR = '/kaggle/input/uav-fix5'
    sys.path.insert(0, MODULE_DIR)
else:
    MODULE_DIR = os.path.join(os.path.dirname(os.getcwd()), 'fixes-2')
    sys.path.insert(0, MODULE_DIR)
print('Module dir:', MODULE_DIR)

In [ ]:
import math, random, time, copy
import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.distributions import Normal
import matplotlib.pyplot as plt

from uav_env_5 import (
    UAVFieldEnv,
    N_UAVS, N_SECTORS, OBS_SIZE, JOINT_SIZE,
    ENV_OBS_DIM, GRID_ROWS, GRID_COLS, N_SECTORS as NS,
)
from uav_env_5 import N_SECTORS

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## Mode Definitions
Reward scaling factors per mode and a hard-coded selector that simulates the LLM
mode-selection output using available crop-disease telemetry.

| Mode       | detection | task_completion | safety |
|------------|-----------|-----------------|--------|
| Aggressive | 3.0       | 2.0             | 0.5    |
| Balanced   | 1.5       | 1.0             | 1.0    |
| Cautious   | 0.5       | 0.5             | 2.0    |

In [ ]:
MODES = {
    'aggressive': {'detection': 3.0, 'task': 2.0, 'safety': 0.5},
    'balanced':   {'detection': 1.5, 'task': 1.0, 'safety': 1.0},
    'cautious':   {'detection': 0.5, 'task': 0.5, 'safety': 2.0},
}
MODE_NAMES = list(MODES.keys())


def build_selector_context(env):
    """Telemetry context used by the hard-coded LLM selector analogue."""
    infected_frac = float(env.true_status.sum()) / N_SECTORS
    avg_energy = float(np.mean(env.energy)) / 150.0
    weather_risk = max(
        float(env.wind_speed) / 12.0,
        float(env.humidity) / 100.0,
        float(env.season_mult) / 1.5,
    )
    return {
        'infected_frac': infected_frac,
        'avg_energy': avg_energy,
        'wind_speed': float(env.wind_speed),
        'humidity': float(env.humidity),
        'season_mult': float(env.season_mult),
        'weather_risk': weather_risk,
    }


def select_mode(context: dict) -> str:
    """Hard-coded threshold selector used in place of a fine-tuned LLM."""
    # Safety first under low battery or turbulent wind.
    if context['avg_energy'] < 0.35 or context['wind_speed'] >= 8.5:
        return 'cautious'
    # Escalate containment when disease pressure or weather-driven spread risk is high.
    if (context['infected_frac'] >= 0.03 or
            context['humidity'] >= 75.0 or
            context['season_mult'] >= 1.15):
        return 'aggressive'
    # Use a middle policy for moderate environmental risk.
    if context['weather_risk'] >= 0.55:
        return 'balanced'
    return 'cautious'


def scale_reward(raw_reward: float, comp: dict, mode_params: dict) -> float:
    """Replace default component weights with mode-specific scaling."""
    r = raw_reward
    # Swap discovery_bonus weight
    r -= comp.get('discovery_bonus', 0.0)
    r += mode_params['detection'] * comp.get('discovery_bonus', 0.0)
    # Swap crash_penalty weight (was subtracted in env)
    r += comp.get('crash_penalty', 0.0)
    r -= mode_params['safety'] * comp.get('crash_penalty', 0.0)
    # Swap safe_return_bonus weight
    r -= comp.get('safe_return_bonus', 0.0)
    r += mode_params['task'] * comp.get('safe_return_bonus', 0.0)
    return r

## Network Architectures
Imported from `networks_5.py` — identical to the main MAPPO baseline.
- `SectorAttentionActor`: 2-layer Transformer over 100 sector tokens + global embed.
- `CriticNetwork`: shared trunk + 4 per-UAV linear heads (one value head per UAV).

In [ ]:
from networks_5 import SectorAttentionActor, CriticNetwork

# One actor per mode; all share a single centralized critic
actors     = {m: SectorAttentionActor().to(DEVICE) for m in MODE_NAMES}
critic     = CriticNetwork().to(DEVICE)
actor_opts = {m: Adam(actors[m].parameters(), lr=3e-4) for m in MODE_NAMES}
critic_opt = Adam(critic.parameters(), lr=1e-3)
print({m: sum(p.numel() for p in actors[m].parameters()) for m in MODE_NAMES})
print('Critic params:', sum(p.numel() for p in critic.parameters()))

## Hyperparameters

In [ ]:
EPISODES     = 400
GAMMA_PPO    = 0.99
GAE_LAMBDA   = 0.95
K_EPOCHS     = 2
MINI_BATCH   = 128
CLIP_EPS     = 0.2
MAX_GRAD     = 0.5
REWARD_CLIP  = 50.0
KL_THRESH    = 0.02
ENTROPY_START = 0.01
ENTROPY_END   = 0.001

def entropy_coeff(ep):
    t = ep / max(EPISODES - 1, 1)
    return ENTROPY_START * (1 - t) + ENTROPY_END * t

## Rollout Buffer & GAE

In [ ]:
class RolloutBuffer:
    def __init__(self):
        self.obs        = [[] for _ in range(N_UAVS)]
        self.actions    = [[] for _ in range(N_UAVS)]
        self.log_probs  = [[] for _ in range(N_UAVS)]
        self.rewards    = [[] for _ in range(N_UAVS)]
        self.values     = []
        self.dones      = []
        self.alive      = [[] for _ in range(N_UAVS)]

    def clear(self):
        self.__init__()


def compute_gae(rewards_u, values_tensor, dones, alive_u):
    T = len(rewards_u)
    adv = torch.zeros(T, device=DEVICE)
    gae = 0.0
    vals = values_tensor.detach()
    for t in reversed(range(T)):
        mask  = 1.0 - float(dones[t])
        next_v = vals[t + 1] if t + 1 < len(vals) else 0.0
        delta = rewards_u[t] + GAMMA_PPO * next_v * mask - vals[t]
        gae   = delta + GAMMA_PPO * GAE_LAMBDA * mask * gae
        adv[t] = gae
    ret = adv + vals[:T]
    alive_t = torch.tensor(alive_u, dtype=torch.float32, device=DEVICE)
    if alive_t.sum() > 1:
        adv = (adv - adv[alive_t.bool()].mean()) / (adv[alive_t.bool()].std() + 1e-8)
    return adv, ret

In [ ]:
def ppo_update(mode, buf, ep):
    actor = actors[mode]
    a_opt = actor_opts[mode]
    ent_c = entropy_coeff(ep)

    T = len(buf.dones)
    all_rews = np.concatenate([buf.rewards[u] for u in range(N_UAVS)])
    scale    = max(float(np.std(all_rews)), 1.0)

    joint_obs_t = torch.tensor(
        np.stack([np.concatenate([buf.obs[u][t] for u in range(N_UAVS)])
                  for t in range(T)]),
        dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        all_vals = critic(joint_obs_t)  # (T, N_UAVS)

    adv_list, ret_list = [], []
    for u in range(N_UAVS):
        rews_norm = [float(np.clip(r, -REWARD_CLIP, REWARD_CLIP)) / scale
                     for r in buf.rewards[u]]
        adv_u, ret_u = compute_gae(
            rews_norm, all_vals[:, u], buf.dones, buf.alive[u])
        adv_list.append(adv_u)
        ret_list.append(ret_u)

    obs_t    = [torch.tensor(np.array(buf.obs[u]),     dtype=torch.float32, device=DEVICE)
                for u in range(N_UAVS)]
    act_t    = [torch.tensor(np.array(buf.actions[u]), dtype=torch.float32, device=DEVICE)
                for u in range(N_UAVS)]
    lp_old_t = [torch.tensor(np.array(buf.log_probs[u]), dtype=torch.float32, device=DEVICE)
                for u in range(N_UAVS)]
    alive_t  = [torch.tensor(buf.alive[u], dtype=torch.float32, device=DEVICE)
                for u in range(N_UAVS)]

    idx = np.arange(T)
    for _ in range(K_EPOCHS):
        np.random.shuffle(idx)
        for start in range(0, T, MINI_BATCH):
            mb = idx[start:start + MINI_BATCH]
            mb_t = torch.tensor(mb, device=DEVICE)

            # Actor loss (mode-specific actor only)
            a_loss = torch.tensor(0.0, device=DEVICE)
            kl_sum = 0.0
            for u in range(N_UAVS):
                lp_new, ent = actor.get_log_prob_entropy(
                    obs_t[u][mb_t], act_t[u][mb_t])
                ratio  = torch.exp(lp_new - lp_old_t[u][mb_t])
                adv_mb = adv_list[u][mb_t]
                surr1  = ratio * adv_mb
                surr2  = torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS) * adv_mb
                mb_alive = alive_t[u][mb_t]
                pol_loss = (-torch.min(surr1, surr2) * mb_alive).sum()
                n_alive  = mb_alive.sum().clamp(min=1)
                a_loss  += pol_loss / n_alive - ent_c * ent
                with torch.no_grad():
                    kl_sum += (lp_old_t[u][mb_t] - lp_new).mean().item()

            if kl_sum / N_UAVS > KL_THRESH:
                break

            a_opt.zero_grad()
            (a_loss / N_UAVS).backward()
            nn.utils.clip_grad_norm_(actor.parameters(), MAX_GRAD)
            a_opt.step()

            # Critic loss (shared)
            pred_v = critic(joint_obs_t[mb_t])  # (mb, N_UAVS)
            c_loss = sum(
                nn.functional.mse_loss(pred_v[:, u], ret_list[u][mb_t])
                for u in range(N_UAVS)
            ) / N_UAVS
            critic_opt.zero_grad()
            c_loss.backward()
            nn.utils.clip_grad_norm_(critic.parameters(), MAX_GRAD)
            critic_opt.step()

## Training Loop

In [ ]:
env = UAVFieldEnv(seed=42)

ep_rewards   = []
ep_infected  = []  # infected fraction at end of episode
ep_diagnosed = []  # fraction of sectors ever diagnosed
ep_crashes   = []
ep_modes     = []
ep_contexts  = []
mode_counts  = {m: 0 for m in MODE_NAMES}

t0 = time.time()
for ep in range(EPISODES):
    obs_list = env.reset()

    # Mode selection: hard-coded telemetry thresholds simulating the LLM selector
    selector_ctx  = build_selector_context(env)
    mode          = select_mode(selector_ctx)
    mode_params   = MODES[mode]
    actor         = actors[mode]
    mode_counts[mode] += 1
    ep_modes.append(mode)
    ep_contexts.append(selector_ctx)

    buf = RolloutBuffer()
    ep_r = 0.0
    crashes_this_ep = 0

    for _ in range(env.total_steps):
        obs_tensors = [torch.tensor(obs_list[u], dtype=torch.float32, device=DEVICE)
                       for u in range(N_UAVS)]

        actions_np = []
        log_probs  = []
        for u in range(N_UAVS):
            if env.crashed[u]:
                actions_np.append(np.zeros(2, dtype=np.float32))
                log_probs.append(torch.tensor(0.0, device=DEVICE))
            else:
                act, lp = actor.get_action(obs_tensors[u])
                actions_np.append(act)
                log_probs.append(lp.squeeze())

        next_obs, raw_rewards, done, info = env.step(actions_np)

        # Mode-specific reward scaling
        comps = info['reward_components']
        rewards = [scale_reward(raw_rewards[u], comps[u], mode_params)
                   for u in range(N_UAVS)]

        joint_obs_np = np.concatenate(obs_list)
        for u in range(N_UAVS):
            buf.obs[u].append(obs_list[u])
            buf.actions[u].append(actions_np[u])
            buf.log_probs[u].append(log_probs[u].item())
            buf.rewards[u].append(rewards[u])
            buf.alive[u].append(0.0 if env.crashed[u] else 1.0)
        buf.dones.append(done)

        ep_r += sum(raw_rewards)
        crashes_this_ep += sum(info['newly_crashed'])
        obs_list = next_obs
        if done:
            break

    ppo_update(mode, buf, ep)

    ep_rewards.append(ep_r)
    ep_infected.append(float(env.true_status.sum()) / N_SECTORS)
    ep_diagnosed.append(float(env.ever_diagnosed.sum()) / N_SECTORS)
    ep_crashes.append(crashes_this_ep)

    if (ep + 1) % 50 == 0:
        elapsed = time.time() - t0
        print(f'Ep {ep+1:4d}/{EPISODES} | mode={mode:10s} | '
              f'reward={ep_r:8.1f} | inf={ep_infected[-1]:.2f} '
              f'diag={ep_diagnosed[-1]:.2f} | crashes={crashes_this_ep} '
              f'| {elapsed:.0f}s')

print('\nMode selection counts:', mode_counts)

## Metrics & Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Pareto Multi-Mode MAPPO — Training Curves', fontsize=14)

def smooth(x, w=20):
    return np.convolve(x, np.ones(w)/w, mode='valid')

axes[0,0].plot(smooth(ep_rewards), label='smoothed')
axes[0,0].set_title('Episode Reward'); axes[0,0].set_xlabel('Episode')

axes[0,1].plot(smooth(ep_infected), color='red', label='smoothed')
axes[0,1].set_title('Disease Fraction at End of Episode')
axes[0,1].axhline(0.35, color='gray', linestyle='--', label='endemic baseline')
axes[0,1].legend()

axes[1,0].plot(smooth(ep_diagnosed), color='green')
axes[1,0].set_title('Diagnosed Fraction per Episode')

axes[1,1].plot(smooth(ep_crashes), color='orange')
axes[1,1].set_title('Crashes per Episode')

plt.tight_layout()
plt.savefig('pareto_training.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved pareto_training.png')

In [ ]:
# Summary statistics (last 50 episodes)
last = slice(-50, None)
print('=== Pareto MAPPO — Final 50 Episodes ===')
print(f'Mean reward       : {np.mean(ep_rewards[last]):.2f}')
print(f'Mean disease frac : {np.mean(ep_infected[last]):.3f}')
print(f'Mean diag frac    : {np.mean(ep_diagnosed[last]):.3f}')
print(f'Mean crashes/ep   : {np.mean(ep_crashes[last]):.2f}')
print(f'Mode distribution : {mode_counts}')

In [ ]:
import os
SAVE_DIR = '/kaggle/working' if ON_KAGGLE else '.'
for m in MODE_NAMES:
    path = os.path.join(SAVE_DIR, f'pareto_actor_{m}.pt')
    torch.save(actors[m].state_dict(), path)
    print('Saved', path)
torch.save(critic.state_dict(), os.path.join(SAVE_DIR, 'pareto_critic.pt'))
print('Saved pareto_critic.pt')